# Plan noise vs plan-first prompting for code (Qwen3-8B) — demo

**The question.** Ask Qwen3-8B to code a classic algorithm task and it nearly always writes the same *default*
algorithm: a full sort for the k-th smallest element, Dijkstra for a weighted shortest path. Which "knob" actually buys
**correct algorithmic diversity**? This experiment (iteration 5) makes two pre-registered claims:

- **Claim F — is temperature the creativity knob?** On the shortest-path task T2 with thinking on, top-k 10 and the
  same token seed, moving the temperature over {0.3, 0.7, 1.0, 1.3} changes the final algorithm on **0 of 20 seeds**
  (Clopper-Pearson one-sided 95% upper bound 0.14). All 100 traces are heap-based Dijkstra.
- **Claim C — plan noise vs plan-first prompting.** *Plan noise* (PN) adds an isotropic random direction to the
  hidden states at the task-specification positions of decoder blocks 12–23, during the prompt prefill only
  (strength α ∈ {1, 2}). It is compared with **text-level plan samplers** that make the model state a plan first
  (IFG, PlanSearch-lite, enumerate-and-pick EAP, a base-model-written intent BINT) and with **departure-matched
  decoding** (DMD: a temperature on generated tokens 1–32, tuned so that its per-token KL equals PN's). Each knob is
  priced on a *randomisation hull*: **H5** = the largest correct-only Hill-1 diversity reachable while losing at most
  5 pass points. Committed rule: SUPPORTED if H5(PN) ≥ the best text sampler's H5 on ≥ 3 of the 5 decisive tasks.
  Result: **DISCONFIRMED** (PN wins 1 of 5; the base-written intent BINT wins three).

**What this notebook runs.** The original `method.py` is a stage orchestrator. Its generation stages need Qwen3-8B on
a 24 GB GPU for hours (8,576 HF rows plus a vLLM cell), so they are only listed here. The notebook re-runs the
**Claim-C analysis** of the `analyze` stage (`src/analysis_c.py` with the hull code `src/hull_full.py` and
`src/hull_exp20.py`, copied with minimal changes) on the real labelled programs of two decisive tasks: **T03**
(k-th smallest element) and **T2** (weighted shortest path), 50 slots × 15–16 knob arms each, 1,550 labelled
programs. It then re-draws the original figures (`src/figures5.py`).

At the original Monte-Carlo settings the point prices (H5 / P15 / P20) match the committed values of the full run
exactly. Only the slot bootstrap runs with fewer replicates than the original 2,000 (see the config cell).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install (method.py logs with it)
_pip('loguru==0.7.3')

# numpy, scipy, matplotlib, pandas — pre-installed on Colab, install locally only (Colab's exact versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'pandas==2.2.2')

In [ ]:
from __future__ import annotations

# method.py
import subprocess
import sys
import time
from pathlib import Path

from loguru import logger

# src/analysis_c.py, src/hull_full.py, src/hull_exp20.py
import itertools
import json
import math
import multiprocessing as mp
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from typing import Any

import numpy as np

# src/figures5.py (without matplotlib.use("Agg"), so the figures render inline) + pandas for the result tables
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-295f8e-prompt-bans-substitute-not-just-subtract/fork/run_jSassd-YWLVC/round-5/experiment-27/demo/mini_demo_data.json"
import http.client
import json
import urllib.request
from pathlib import Path

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            return json.loads(response.read().decode())
    # Network errors, timeouts, a placeholder URL, bad JSON, a cut-off response.
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
_ex = data["datasets"][0]["examples"]
print(data["datasets"][0]["dataset"], "-", len(_ex), "slots:", dict(Counter(e["metadata_task"] for e in _ex)))
print("one slot (T03, W1, k=0): TWIN ->", _ex[0]["output"], "| PN_a2 ->", _ex[0]["predict_PN_a2"],
      "| EAP ->", _ex[0]["predict_EAP"], "| NOVELTY ->", _ex[0]["predict_NOVELTY"])
for t, p in data["metadata"]["prompts_W1"].items():
    print(f"\n--- {t} ({data['metadata']['task_names'][t]}): the TWIN prompt, template W1 ---\n{p['twin']}")

## Configuration

All tunable parameters of the analysis. The values below are the demo's; the original values of the full run are in
the comments. The point prices use the original Monte-Carlo sizes (so H5 / P15 / P20 reproduce the committed numbers
exactly). The bootstrap is the expensive part. The original ran 2,000 replicates per decisive task on 12 worker
processes; this demo runs 200. Setting `N_BOOT_D5 = 2000` reproduces the committed bootstrap CIs and P(PN ≥ best)
for these two tasks exactly. It takes about 0.1 s per replicate per task on one CPU, so ≈ 7–8 minutes.

In [ ]:
# ---- tunable parameters (demo values; original values of the full run in the comments) ----
N_DRAWS_PRICE = 1000  # Monte-Carlo draws of the randomisation hull for the point prices (original: 1000)
N_RAREFY = 1000       # subsamples of the rarefied Hill-1 of each cell (original: 1000)
N_BOOT_D5 = 200       # slot-bootstrap replicates per decisive task (original: 2000)
N_BOOT_ND = 500       # replicates per non-decisive task (original: 500; no ND10 task is in the demo data)
BOOT_RAREFY = 200     # hull draws inside each bootstrap replicate (original: 200)
N_WORKERS = 1         # bootstrap worker processes (original: 12; 1 = run in this process, see the bootstrap cell)

## 1. The pipeline: `method.py`

`method.py` runs each stage as a subprocess (`src/*.py`), in the order below, and every stage resumes from what is
already on disk. The vLLM literal cell runs in its own environment (`.venv_vllm`). What this notebook does with each
stage:

| Stage | What it does | Here |
|---|---|---|
| `s0`, `prompts` | precision rule, input hashes, seed registry, labeller re-check; the 430 stage-1 prompts | two W1 prompts printed above |
| `commit0F`, `commit0C`, `commit1` | sha256 pre-registration chain (specs, analysis code, DMD temperatures) | – |
| `literal`, `label_f` | Claim F: 100 thinking-on traces on vLLM (FP8), labelled by the frozen labellers | committed result re-checked at the end |
| `gpu` | base intents, DMD calibration, engine check, the knob queue (HF bf16, exact inverse-CDF common random numbers) | rows are the demo data |
| `label`, `audit` | mechanism labels (runtime trace + AST, blind judge on disagreement); lexicon / style / novelty audits | labels are the demo data |
| `analyze` | `analysis_f` (Claim F), `analysis_ec` (engine check), **`analysis_c` (Claim C)** | **`analysis_c` re-run below** |
| `figures`, `outputs`, `rederive` | F1–F6, `method_out.json`, independent re-derivation | F1–F6 re-drawn |

The cell defines the orchestrator verbatim, then prints the stage plan instead of calling `main()`.

In [ ]:
ROOT = Path.cwd()  # notebook: was Path(__file__).resolve().parent
PY = str(ROOT / ".venv" / "bin" / "python")
PYV = str(ROOT / ".venv_vllm" / "bin" / "python")
# (ROOT / "logs").mkdir(exist_ok=True)  # notebook: no logs/ folder, stdout only
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
# logger.add(str(ROOT / "logs" / "method.log"), rotation="30 MB", level="DEBUG")  # notebook: no file sink

STAGES: dict[str, list[list[str]]] = {
    "s0": [[PY, "src/s0.py", "all"]],
    "prompts": [[PY, "src/prompts5.py"]],
    "commit0F": [[PY, "src/prereg5.py", "commit0F"]],
    "literal": [[PYV, "src/vllm_literal.py", "all"]],
    "label_f": [[PY, "src/label_worker_v4.py", "--raw", "data/literal_samples.jsonl", "--out",
                 "data/literal_labels.jsonl", "--name", "label_literal"]],
    "gpu": [[PY, "src/gen5.py", "base,tests,calib,ec,knobs"]],
    "commit0C": [[PY, "src/prereg5.py", "commit0C"]],
    "commit1": [[PY, "src/prereg5.py", "commit1"]],
    "label": [[PY, "src/label5.py"],
              [PY, "src/label_worker_v4.py", "--raw", "data/engine_check_raw.jsonl", "--out",
               "data/engine_check_labels.jsonl", "--name", "label_ec"]],
    "audit": [[PY, "src/audit5.py", "all"]],
    "analyze": [[PY, "src/analysis_f.py"], [PY, "src/analysis_ec.py"], [PY, "src/analysis_c.py"]],
    "figures": [[PY, "src/figures5.py"]],
    "outputs": [[PY, "src/outputs5.py"]],
    "rederive": [[PY, "audit/rederive.py"]],
}
ORDER = ["s0", "prompts", "commit0F", "literal", "label_f", "gpu", "commit0C", "commit1", "label", "audit",
         "analyze", "figures", "outputs", "rederive"]


@logger.catch(reraise=True)
def main() -> None:
    want = ORDER if len(sys.argv) < 2 or sys.argv[1] == "all" else sys.argv[1].split(",")
    for st in want:
        if st not in STAGES:
            raise SystemExit(f"unknown stage {st}; choose from {ORDER}")
        if st == "gpu" and "all" in sys.argv[1:]:
            logger.warning("the gpu stage blocks on commit0C/commit1: run commit0C and commit1 from a second shell "
                           "once calibration has finished (results/calib_done.flag)")
        for cmd in STAGES[st]:
            t0 = time.time()
            logger.info(f"[{st}] {' '.join(cmd[1:])}")
            r = subprocess.run(cmd, cwd=str(ROOT))
            if r.returncode != 0:
                logger.error(f"[{st}] failed with exit code {r.returncode}")
                raise SystemExit(r.returncode)
            logger.info(f"[{st}] done in {time.time() - t0:.0f}s")


# main() would launch every stage as a subprocess; the GPU stages need Qwen3-8B on a 24 GB GPU and the src/ tree.
# Here we only print the plan (the same log lines main() writes) and re-run the analyze stage's Claim C below.
for st in ORDER:
    for cmd in STAGES[st]:
        logger.info(f"[{st}] {' '.join(cmd[1:])}")

## 2. Frozen constants (`src/cfg5.py`)

- **Tasks.** D5 = the five decisive tasks (N = 50 slots per cell: templates W1–W5 × k 0–9). ND10 = ten more tasks
  (N = 30), not in the demo data.
- **Arms.** Every arm decodes the same task at the same slot, and all arms of a slot share the code-stage stream of
  uniforms (common random numbers), so arms can be compared slot by slot.
  - TWIN: the reference arm at T 1.0. Its prompt is the task preceded by a neutral "twin" sentence ("Do not use
    man and the house for this task.", printed above). PLAIN: the same prompt without the twin sentence.
  - PN_a1 / PN_a2: plan noise at α 1 / 2 (`PN_BLOCKS` 12–23). PNPLAIN_a1: plan noise on the plain prompt.
  - IFG_hot / IFG_paper: the intent is sampled hot and the code cool.
  - BINT: Qwen3-8B-Base writes the intent. PSL: PlanSearch-lite. EAP: enumerate-and-pick.
  - DMD_a1 / DMD_a2: a temperature on tokens 1–32, matched to PN's KL.
  - T1.3, TOPK10, NOVELTY, SEMSEED: the earlier iterations' knobs, rerun.
- **Classes.** A class is the set of arms that the randomisation hull may mix. TWIN always comes first: it is the
  zero-cost operating point.

In [ ]:
# src/cfg5.py — the parts the Claim-C analysis uses (verbatim)
D5 = ("T03", "T10", "T2", "T05", "T1")
ND10 = ("T04", "T06", "T08", "T11", "T13", "T14", "R1", "R2", "R3", "R6")
TASKS15 = D5 + ND10

# frozen sampler instructions (v2 = after the one allowed wording revision, deviation D-pilot)
IFG_INSTR = ("Before writing any code, state in one sentence the algorithm you will use, on a single line that "
             "starts with Intent:. Then write the code.")
# Wording v2 (deviation D-pilot, the plan's one allowed revision before commit #1): the v1 stage-1 sentences
# ("Do not write code yet. ...") lost to the specification's final format line on 0-100% of pilot rows (the model
# wrote the code block at once), so the step structure is stated and the format line is deferred explicitly.
PSL_A_V1 = ("Do not write code yet. First list exactly 5 short, distinct observations about this problem that could "
            "help solve it, numbered 1 to 5.")
EAP_A_V1 = ("Do not write code yet. List the distinct algorithms that correctly solve this task, one per line, "
            "formatted as <number>. <algorithm name>: <one-line idea>. List at most 8.")
PSL_A = ("This is step 1 of 3: do not write any code in this step, and ignore the code-block instruction at the end "
         "of the task for now (you will write the code in a later step). First list exactly 5 short, distinct "
         "observations about this problem that could help solve it, numbered 1 to 5.")
EAP_A = ("This is step 1 of 2: do not write any code in this step, and ignore the code-block instruction at the end "
         "of the task for now (you will write the code in step 2). List the distinct algorithms that correctly solve "
         "this task, one per line, formatted as <number>. <algorithm name>: <one-line idea>. List at most 8.")
PSL_B_V1 = ("Combine observations {i} and {j} into a short natural-language plan (at most 5 sentences) for an "
            "algorithm that solves the problem. Do not write code yet.")
PSL_B = ("Step 2 of 3: combine observations {i} and {j} into a short natural-language plan (at most 5 sentences) for "
         "an algorithm that solves the problem. Do not write any code yet.")
PSL_C = "Now implement your plan in Python with the required function signature. Output only the code in one python code block."
EAP_B = "Implement approach {i}: {name}. Output only the code in one python code block."
INTENT_PREFIX = "Intent:"

# ------------------------------------------------------------------------------------------------ arms (Claim C)
# kind: prompt kind; T: code-stage temperature; extra per-family fields. 'decode': shared | pn | leftpad | multi
ARMS5: dict[str, dict[str, Any]] = {
    "TWIN": {"kind": "twin", "T": 1.0, "family": "REF", "decode": "shared"},
    "PLAIN": {"kind": "plain", "T": 1.0, "family": "REF_PLAIN", "decode": "shared"},
    "PN_a1": {"kind": "twin", "T": 1.0, "alpha": 1.0, "family": "PN", "decode": "pn"},
    "PN_a2": {"kind": "twin", "T": 1.0, "alpha": 2.0, "family": "PN", "decode": "pn"},
    "PNPLAIN_a1": {"kind": "plain", "T": 1.0, "alpha": 1.0, "family": "PNPLAIN", "decode": "pn"},
    "IFG_hot": {"kind": "ifg", "T": 0.7, "t_intent": 1.3, "family": "IFG", "decode": "shared"},
    "IFG_paper": {"kind": "ifg", "T": 0.60, "t_intent": 0.73, "family": "IFG", "decode": "shared"},
    "BINT": {"kind": "ifg", "T": 0.7, "t_intent": 1.3, "family": "BINT", "decode": "leftpad"},
    "PSL": {"kind": "psl", "T": 0.9, "top_p": 0.95, "family": "PSL", "decode": "multi"},
    "EAP": {"kind": "eap", "T": 1.0, "family": "EAP", "decode": "multi"},
    "DMD_a1": {"kind": "twin", "T": 1.0, "dmd_alpha": 1.0, "family": "DMD", "decode": "shared"},
    "DMD_a2": {"kind": "twin", "T": 1.0, "dmd_alpha": 2.0, "family": "DMD", "decode": "shared"},
    # P3 reruns (bf16 engine: iteration-4 FP8 rows are not reusable slot for slot)
    "T1.3": {"kind": "twin", "T": 1.3, "family": "TEMP", "decode": "shared"},
    "TOPK10": {"kind": "twin", "T": 1.0, "top_k": 10, "family": "TOPK10", "decode": "shared"},
    "NOVELTY": {"kind": "novelty", "T": 1.0, "family": "NOVELTY", "decode": "shared"},
    "SEMSEED": {"kind": "semseed", "T": 1.0, "family": "SEMSEED", "decode": "shared"},
}
PN_BLOCKS = list(range(12, 24))

# ------------------------------------------------------------------------------------------------ analysis (frozen)
DEPTH = 25
# N_RAREFY, N_BOOT_D5, N_BOOT_ND, BOOT_RAREFY: set in the config cell
COST_BOUND = 0.05
HILL_TARGETS = (1.5, 2.0)
CLAIM_C_MIN_WINS = 3
UNRESOLVED_BAND = (0.2, 0.8)
MOVE_EXCESS = 0.10
MOVE_FISHER_P = 0.05
BOOT_SEED5 = 20261021
TEXT_SAMPLERS = ("IFG", "PSL", "EAP", "BINT")
CLASS_SETTINGS: dict[str, list[str]] = {
    "PN": ["TWIN", "PN_a1", "PN_a2"],
    "PN+": ["TWIN", "PN_a1", "PN_a2", "PLAIN", "PNPLAIN_a1"],
    "IFG": ["TWIN", "IFG_hot", "IFG_paper"],
    "IFG_hot_only": ["TWIN", "IFG_hot"],
    "PSL": ["TWIN", "PSL"],
    "EAP": ["TWIN", "EAP"],
    "BINT": ["TWIN", "BINT"],
    "DMD": ["TWIN", "DMD_a1", "DMD_a2"],
    "TEMP": ["TWIN", "T1.3"],
    "TOPK10": ["TWIN", "TOPK10"],
    "NOVELTY": ["TWIN", "NOVELTY"],
    "SEMSEED": ["TWIN", "SEMSEED"],
}

## 3. Rarefied Hill-1 and the CRN hull sampler (`src/hull_exp20.py`)

- **Diversity measure.** The *correct-only Hill-1 at depth 25* is the effective number of distinct algorithms among 25
  correct programs: exp(Shannon entropy) of their label composition, averaged over subsamples.
  - 1.0 means every correct program uses the default algorithm.
  - 2.0 is as diverse as an even split between two algorithms.
- **Mixing arms.** The *randomisation hull* mixes the settings of a class with weights w on the simplex.
  - Pass cost of a mixture: cost(w) = pass_TWIN − Σ_j w_j · pass_j.
  - Hill(w) is estimated by a two-stage draw: allocate the 25 programs to settings ~ Multinomial(25, π), with
    π_j ∝ w_j · pass_j, then take that many correct programs of each setting.
  - `HullSampler` pre-draws all randomness once, so every w is evaluated on the same common random numbers.

exp20's legacy `hull_prices` and C1-site rule in the same file are not used by this artifact and are omitted.

In [ ]:
# src/hull_exp20.py (verbatim)
DEPTH = 25
GRID_STEP = 0.05
COST_BOUND = 0.05
HILL_TARGETS = (1.5, 2.0)


def hill1(counts: np.ndarray) -> np.ndarray:
    """exp(Shannon entropy) along the last axis of a count array."""
    c = np.asarray(counts, dtype=np.float64)
    n = c.sum(-1, keepdims=True)
    p = np.where(n > 0, c / np.maximum(n, 1e-12), 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -np.where(p > 0, p * np.log(p), 0.0).sum(-1)
    return np.exp(h)


def rarefied_hill(labels: list[str], depth: int = DEPTH, n_sub: int = 1000, seed: int = 0) -> dict[str, Any]:
    """Rarefied Hill-1; cells with fewer than `depth` correct programs report Hill at their own depth (flagged)."""
    n = len(labels)
    if n == 0:
        return {"hill": None, "depth": 0, "flag_own_depth": True, "n": 0}
    cats, inv = np.unique(np.array(labels), return_inverse=True)
    counts = np.bincount(inv, minlength=len(cats))
    m = min(depth, n)
    rng = np.random.default_rng(seed)
    sub = rng.multivariate_hypergeometric(counts, m, size=n_sub)
    return {"hill": float(hill1(sub).mean()), "depth": int(m), "flag_own_depth": bool(m < depth), "n": int(n)}


def simplex_grid(k: int, step: float = GRID_STEP, max_nonzero: int = 3) -> np.ndarray:
    m = int(round(1 / step))
    pts = []
    for comb in itertools.product(range(m + 1), repeat=k - 1):
        s = sum(comb)
        if s <= m:
            w = list(comb) + [m - s]
            if sum(1 for x in w if x > 0) <= max_nonzero:
                pts.append(w)
    return np.array(pts, dtype=np.float64) / m


class HullSampler:
    """Pre-draws the CRN randomness for one (task, site): per draw d, 25 uniforms (multinomial allocation by inverse
    CDF on pi(w)) and, per setting j, an ordered sequence of 25 correct programs (a random permutation, then draws with
    replacement beyond N_correct_j). C[j][d, n, :] = label counts among the first n programs of setting j in draw d."""

    def __init__(self, settings: list[dict[str, Any]], n_draws: int = 1000, depth: int = DEPTH, seed: int = 0):
        self.settings = settings
        self.depth = depth
        rng = np.random.default_rng(seed)
        cats = sorted({l for s in settings for l in s["correct_labels"]})
        self.cats = cats
        ci = {c: i for i, c in enumerate(cats)}
        K = max(len(cats), 1)
        self.U = rng.random((n_draws, depth))
        self.C = []
        self.with_replacement = []
        for s in settings:
            labs = np.array([ci[l] for l in s["correct_labels"]], dtype=np.int64)
            C = np.zeros((n_draws, depth + 1, K), dtype=np.int32)
            if len(labs):
                seqs = np.empty((n_draws, depth), dtype=np.int64)
                for d in range(n_draws):
                    if len(labs) >= depth:
                        seqs[d] = rng.permutation(labs)[:depth]
                    else:
                        extra = rng.choice(labs, size=depth - len(labs), replace=True)
                        seqs[d] = np.concatenate([rng.permutation(labs), extra])
                oh = np.zeros((n_draws, depth, K), dtype=np.int32)
                np.put_along_axis(oh, seqs[..., None], 1, axis=2)
                C[:, 1:] = np.cumsum(oh, axis=1)
            self.C.append(C)
            self.with_replacement.append(bool(0 < len(labs) < depth))
        self.pass_ = np.array([s["pass"] for s in settings], dtype=np.float64)
        self.n_correct = np.array([len(s["correct_labels"]) for s in settings])

    def hill_of(self, w: np.ndarray) -> float:
        a = w * self.pass_
        if a.sum() <= 0:
            return float("nan")
        pi = a / a.sum()
        cdf = np.cumsum(pi)
        cdf[-1] = 1.0
        j_of = np.searchsorted(cdf, self.U, side="right").clip(max=len(pi) - 1)  # [D, depth]
        comp = 0
        for j in range(len(pi)):
            nj = (j_of == j).sum(1)
            if pi[j] > 0:
                comp = comp + np.take_along_axis(self.C[j], nj[:, None, None], axis=1)[:, 0]
        return float(hill1(comp).mean())

    def cost_of(self, w: np.ndarray) -> float:
        return float(self.pass_[0] - (w * self.pass_).sum())

## 4. Full-simplex hull prices (`src/hull_full.py`)

`hull_prices_full` evaluates Hill(w) and cost(w) on a simplex grid (step 0.01 for ≤ 3 settings, 0.02 for 4, 0.05
for 5), then refines locally around the best feasible point. It returns three prices:

- **H5**: the maximum Hill(w) subject to cost(w) ≤ 0.05.
- **P15 / P20**: the minimum cost that reaches Hill 1.5 / 2.0 (∞ = never reached).
- **Frontier**: the (cost, best Hill so far) frontier, drawn in figure F2.

The edge-restricted sensitivity (`max_nonzero=2`) allows at most two non-zero weights. The file's CPU unit tests
(`unit_tests`, run by `src/tests_cpu.py`; all passed) are omitted.

In [ ]:
# src/hull_full.py (verbatim)
def grid_step(k: int) -> float:
    return 0.01 if k <= 3 else (0.02 if k == 4 else 0.05)


def local_grid(w0: np.ndarray, step: float, radius: float, support_only: bool) -> np.ndarray:
    k = len(w0)
    idx = [j for j in range(k) if (w0[j] > 0 or not support_only)]
    if len(idx) <= 1:
        return w0[None]
    free = idx[:-1]
    offs = np.arange(-radius, radius + 1e-12, step)
    pts = []
    for d in itertools.product(offs, repeat=len(free)):
        w = w0.copy()
        for j, dj in zip(free, d):
            w[j] += dj
        w[idx[-1]] -= sum(d)
        if (w >= -1e-12).all():
            pts.append(np.clip(w, 0, None) / max(np.clip(w, 0, None).sum(), 1e-12))
    return np.array(pts) if pts else w0[None]


def _prices_on(hs: HullSampler, grid: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    costs = np.array([hs.cost_of(w) for w in grid])
    hills = np.array([hs.hill_of(w) for w in grid])
    return costs, hills


def hull_prices_full(settings: list[dict[str, Any]], n_draws: int = 1000, seed: int = 0, step: float | None = None,
                     max_nonzero: int | None = None, refine: bool = True, keep_frontier: bool = False,
                     refine_step: float | None = None) -> dict[str, Any]:
    k = len(settings)
    if k == 1:
        hs = HullSampler(settings, n_draws=n_draws, seed=seed)
        h = hs.hill_of(np.array([1.0]))
        return {"H5": h, "H5_mixture": [1.0], "P15": 0.0 if h >= 1.5 else math.inf,
                "P20": 0.0 if h >= 2.0 else math.inf, "frontier": [[0.0, h]] if keep_frontier else None}
    hs = HullSampler(settings, n_draws=n_draws, seed=seed)
    st = step or grid_step(k)
    grid = simplex_grid(k, st, max_nonzero=max_nonzero or k)
    costs, hills = _prices_on(hs, grid)
    ok = np.isfinite(hills)
    feas = ok & (costs <= COST_BOUND + 1e-12)
    out: dict[str, Any] = {"grid_step": st, "n_grid": int(len(grid)), "max_nonzero": int(max_nonzero or k)}
    if refine and feas.any():
        w0 = grid[feas][int(np.argmax(hills[feas]))]
        rs = refine_step or (0.0025 if k <= 3 else 0.01)
        lg = local_grid(w0, rs, 0.04, support_only=(k > 3))
        if max_nonzero is not None:
            lg = lg[(lg > 1e-12).sum(1) <= max_nonzero]
        if len(lg):
            c2, h2 = _prices_on(hs, lg)
            grid = np.vstack([grid, lg])
            costs = np.concatenate([costs, c2])
            hills = np.concatenate([hills, h2])
            ok = np.isfinite(hills)
            feas = ok & (costs <= COST_BOUND + 1e-12)
        out["n_refine"] = int(len(lg))
    out["H5"] = float(hills[feas].max()) if feas.any() else float("nan")
    out["H5_mixture"] = grid[feas][int(np.argmax(hills[feas]))].tolist() if feas.any() else None
    for tgt in HILL_TARGETS:
        m = ok & (hills >= tgt)
        out[f"P{int(tgt * 10)}"] = float(costs[m].min()) if m.any() else math.inf
    if keep_frontier:
        order = np.argsort(costs)
        fr, best = [], -1.0
        for i in order:
            if ok[i] and hills[i] > best + 1e-9:
                best = hills[i]
                fr.append([round(float(costs[i]), 4), round(float(hills[i]), 4)])
        out["frontier"] = fr
    out["with_replacement_flags"] = hs.with_replacement
    return out

## 5. The labelled rows and each task's default (`src/analysis_c.py`, data section)

- **Rows.** `load_rows` turns every arm's `label|pass` at every slot into one row, as the original does from
  `data/samples_raw.jsonl` and `data/labels.jsonl`. The `pass` flag comes from the hidden tests. The label comes from
  the frozen mechanism labellers (`invalid` = a multi-stage parse failure, which stays in the denominator).
- **Defaults.** `defaults` fixes each decisive task's *default* algorithm: TWIN's modal correct label, if its share is
  ≥ 0.6. Any other correct label counts as **off-default**.
- **Notebook stand-ins.** `src/cfg.py`'s paths and JSON helpers are replaced. Inputs come from `data`, and every
  `results/*.json` the original writes is kept in the in-memory `RESULTS` dict instead.

In [ ]:
# notebook stand-ins for src/cfg.py's paths and JSON helpers (nothing is written to disk)
INPUTS, DATA_DIR, RESULTS_DIR, PRED_DIR, AUDIT_DIR = (ROOT / d for d in ("inputs", "data", "results", "predictions",
                                                                         "audit"))
RESULTS: dict[str, Any] = {}


def read_json(p: Path) -> Any:
    return json.loads(Path(p).read_text())


def write_json(p: Path, obj: Any, indent: int | None = 1) -> None:
    RESULTS[Path(p).name] = obj


# src/analysis_c.py
CLAIM_CLASSES = ["PN", "IFG", "PSL", "EAP", "BINT", "DMD", "IFG_hot_only"]
OTHER_CLASSES = ["TEMP", "TOPK10", "NOVELTY", "SEMSEED"]
LEFTPAD_ARMS = {"BINT", "PSL", "EAP"}


def load_rows(relabel: dict[str, str] | None = None) -> list[dict[str, Any]]:
    # notebook: rebuild the original's two inputs from the knob_slots examples in `data` (one example = one slot
    # with every arm's 'label|pass'): one raw row per (task, arm, slot), replayed in that arm's original file order
    # (metadata_row_order, which the hull's Monte-Carlo draws depend on), and its label keyed by a made-up sample_id.
    # Fields the demo data does not carry (engine path, finish reason, intents, ...) come out as None.
    samples_raw, labs = [], {}
    for e in data["datasets"][0]["examples"]:
        for a in e["metadata_arms_present"]:
            sid = f"{e['metadata_task']}|{a}|{e['metadata_template']}|{e['metadata_slot']}"
            lab_, ok_ = (e["output"] if a == "TWIN" else e["predict_" + a.replace(".", "_")]).rsplit("|", 1)
            samples_raw.append({"sample_id": sid, "task": e["metadata_task"], "arm": a, "template": e["metadata_template"],
                                "slot": e["metadata_slot"], "order": e["metadata_row_order"][a]})
            labs[sid] = {"sample_id": sid, "label": lab_, "passed": ok_ == "pass"}
    samples_raw.sort(key=lambda r: (r["task"], r["arm"], r["order"]))
    out = []
    seen = set()
    for r in samples_raw:  # notebook: was read_jsonl(DATA_DIR / "samples_raw.jsonl")
        sid = r["sample_id"]
        if sid in seen:
            continue
        seen.add(sid)
        l = labs.get(sid)
        if l is None:
            continue
        lab = l.get("label") or "invalid"
        if relabel and l.get("passed") and lab == "other" and l.get("code_sha256") in relabel:
            lab = relabel[l["code_sha256"]]
        out.append({"task": r["task"], "arm": r["arm"], "w": r["template"], "k": int(r["slot"]), "label": lab,
                    "passed": bool(l.get("passed")), "valid": bool(l.get("valid", True)),
                    "engine_path": r.get("engine_path"), "code_sha256": l.get("code_sha256"),
                    "labeller_source": l.get("labeller_source"), "finish_reason": r.get("finish_reason"),
                    "intent_valid": r.get("intent_valid"), "intent_capped": r.get("intent_capped"),
                    "parse_ok": r.get("parse_ok"), "picked_name": r.get("picked_name"),
                    "enum_list": r.get("enum_list"), "unit_ratio": r.get("unit_ratio_banfree_over_ban"),
                    "realised_disp": r.get("realised_disp")})
    return out


def index(rows: list[dict[str, Any]]) -> dict[tuple[str, str], dict[tuple[str, int], dict[str, Any]]]:
    idx: dict[tuple[str, str], dict[tuple[str, int], dict[str, Any]]] = defaultdict(dict)
    for r in rows:
        idx[(r["task"], r["arm"])][(r["w"], r["k"])] = r
    return idx


def exp19_defaults() -> dict[str, dict[str, Any]]:
    out = {}
    rows = []  # notebook: exp19's TWIN rows are not in the demo data (they only set the ND10 defaults)
    by = defaultdict(list)
    for r in rows:
        by[r["task"]].append(r)
    for t, rs in by.items():
        corr = [r["label"] for r in rs if r.get("passed")]
        c = Counter(corr)
        modal, n = c.most_common(1)[0] if c else (None, 0)
        out[t] = {"modal": modal, "share": n / max(len(corr), 1), "N": len(rs), "n_correct": len(corr)}
    return out


def defaults(idx) -> dict[str, dict[str, Any]]:
    e19 = exp19_defaults()
    out = {}
    for t in TASKS15:
        tw = list(idx.get((t, "TWIN"), {}).values())
        corr = [r["label"] for r in tw if r["passed"]]
        c = Counter(corr)
        modal, n = c.most_common(1)[0] if c else (None, 0)
        share = n / max(len(corr), 1)
        if t in D5:
            d = modal if (modal is not None and share >= 0.6) else None
            out[t] = {"default": d, "source": "this TWIN (N 50)", "modal": modal, "share": share, "N": len(tw)}
        else:
            e = e19.get(t, {})
            d = e.get("modal") if (e.get("modal") is not None and e.get("share", 0) >= 0.6) else None
            agree = (modal == d) if d is not None and modal is not None else None
            out[t] = {"default": d if agree is not False else None, "source": "exp19 TWIN (N 50)",
                      "exp19_modal": e.get("modal"), "exp19_share": e.get("share"), "this_modal": modal,
                      "this_share": share, "this_N": len(tw), "confirmed": agree,
                      "flag": None if agree else ("NO DEFAULT (disagreement)" if agree is False else "unconfirmed")}
    return out

## 6. Cell statistics and pricing helpers (`src/analysis_c.py`, stats section)

- **`cell_stats`** describes one (task, arm) cell: pass rate (with a Wilson interval), correct-label composition,
  off-default share, and the rarefied Hill.
- **`mcnemar_exact`** is the exact McNemar test: a binomial test on the discordant slots of two paired arms.
- **`settings_of`** builds a class's settings for the hull, and `price` prices them. Amendment A1 keeps out cells with
  N < 30 on D5.

In [ ]:
def wilson(k: int, n: int, z: float = 1.96) -> list[float | None]:
    if n == 0:
        return [None, None]
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return [max(0.0, c - h), min(1.0, c + h)]


def mcnemar_exact(x: list[int], y: list[int]) -> dict[str, Any]:
    from scipy.stats import binomtest

    b = sum(1 for a, c in zip(x, y) if a == 1 and c == 0)
    c = sum(1 for a, cc in zip(x, y) if a == 0 and cc == 1)
    p = float(binomtest(b, b + c, 0.5).pvalue) if b + c > 0 else 1.0
    return {"n_pairs": len(x), "x_only": b, "y_only": c, "p_exact": p}


def cell_stats(rs: list[dict[str, Any]], default: str | None) -> dict[str, Any]:
    n = len(rs)
    corr = [r for r in rs if r["passed"]]
    labs = [r["label"] for r in corr]
    off = [r for r in corr if default is not None and r["label"] != default]
    rh = rarefied_hill(labs, depth=DEPTH, n_sub=N_RAREFY, seed=0) if labs else {"hill": None, "depth": 0,
                                                                                 "flag_own_depth": True, "n": 0}
    out = {"N": n, "pass": len(corr) / n if n else None, "pass_wilson95": wilson(len(corr), n),
           "n_correct": len(corr), "correct_composition": dict(Counter(labs)),
           "attempted_composition": dict(Counter(r["label"] for r in rs)),
           "invalid_rate": sum(r["label"] == "invalid" for r in rs) / n if n else None,
           "other_rate": sum(r["label"] == "other" for r in rs) / n if n else None,
           "off_default_share": len(off) / n if n else None,
           "correct_only_off_default_share": len(off) / len(corr) if corr else None,
           "hill_rarefied": rh["hill"], "hill_depth": rh["depth"], "hill_own_depth_flag": rh["flag_own_depth"],
           "cap_hits": sum(r["finish_reason"] == "length" for r in rs),
           "engine_paths": dict(Counter(r["engine_path"] for r in rs))}
    if any(r.get("intent_valid") is not None for r in rs):
        iv = [r for r in rs if r.get("intent_valid") is not None]
        out["intent_valid_rate"] = sum(bool(r["intent_valid"]) for r in iv) / len(iv)
        out["intent_capped_rate"] = sum(bool(r.get("intent_capped")) for r in iv) / len(iv)
    if any(r.get("parse_ok") is not None for r in rs):
        out["parse_ok_rate"] = sum(r.get("parse_ok") is not False for r in rs) / n
    return out


MIN_N = {"D5": 30, "ND10": 15}   # amendment A1: incomplete cells (e.g. a dropped arm's stage-100 rows) stay out


def min_n(task: str) -> int:
    return MIN_N["D5"] if task in D5 else MIN_N["ND10"]


def settings_of(idx, task: str, cls: str, slots: list[tuple[str, int]] | None = None) -> list[dict[str, Any]] | None:
    arms = CLASS_SETTINGS[cls]
    out = []
    for a in arms:
        cell = idx.get((task, a))
        if not cell or len(cell) < min_n(task):
            if a == "TWIN":
                return None
            continue
        rs = [cell[s] for s in slots if s in cell] if slots is not None else list(cell.values())
        if not rs:
            continue
        out.append({"arm": a, "correct_labels": [r["label"] for r in rs if r["passed"]],
                    "pass": sum(r["passed"] for r in rs) / len(rs)})
    if len(out) < 2 or out[0]["arm"] != "TWIN":
        return None
    return out


def price(idx, task: str, cls: str, **kw) -> dict[str, Any] | None:
    st = settings_of(idx, task, cls)
    if st is None:
        return None
    res = hull_prices_full(st, **kw)
    res["arms"] = [s["arm"] for s in st]
    return res

## 7. Slot bootstrap (`src/analysis_c.py`, bootstrap section)

Each replicate works as follows:

1. Resample the slots k **within each template**, jointly for every arm of the task. This keeps the common-random-number
   pairing between arms.
2. Re-price every class on the resampled rows: grid 0.05 plus a local refinement at 0.01, with `BOOT_RAREFY` hull
   draws.
3. Re-select every class's best mixture.

The replicate seeds are fixed per (task, replicate index), so the first `N_BOOT_D5` replicates here are exactly the
first `N_BOOT_D5` of the original 2,000.

The only change: with `N_WORKERS = 1` the replicates run in this process, because a spawned worker process cannot import
a function defined in a notebook.

In [ ]:
def _boot_worker(args: tuple) -> list[dict[str, float]]:
    rows, task, classes, seeds = args
    idx = index(rows)
    slots_by_w = defaultdict(set)
    for (t, a), cell in idx.items():
        if t == task:
            for (w, k) in cell:
                slots_by_w[w].add(k)
    out = []
    for sd in seeds:
        rng = np.random.default_rng(sd)
        # resample slots within each template, jointly for every arm; duplicates become distinct pseudo-slots
        mapping = []
        for w in sorted(slots_by_w):
            ks = sorted(slots_by_w[w])
            draw = rng.choice(ks, size=len(ks), replace=True)
            mapping += [(w, int(k)) for k in draw]
        rec: dict[str, float] = {}
        for cls in classes:
            arms = CLASS_SETTINGS[cls]
            st = []
            for a in arms:
                cell = idx.get((task, a))
                if not cell or len(cell) < min_n(task):
                    continue
                rs = [cell[s] for s in mapping if s in cell]
                if not rs:
                    continue
                st.append({"arm": a, "correct_labels": [r["label"] for r in rs if r["passed"]],
                           "pass": sum(r["passed"] for r in rs) / len(rs)})
            if len(st) < 2 or st[0]["arm"] != "TWIN":
                continue
            k = len(st)
            r = hull_prices_full(st, n_draws=BOOT_RAREFY, seed=int(sd % (2 ** 31)), step=0.05 if k <= 4 else 0.1,
                                 refine=True, refine_step=0.01)
            rec[cls] = r["H5"]
            rec[cls + "_P15"] = r["P15"]
            rec[cls + "_P20"] = r["P20"]
        out.append(rec)
    return out


def bootstrap(rows, task: str, classes: list[str], n_boot: int, n_workers: int) -> list[dict[str, float]]:
    """Amendment A3 (computation only): replicates are cached per task in results/boot_cache/<task>.json, keyed by the
    sha256 of the task's (sample_id, arm, slot, label, passed) rows, the class list and n_boot; identical inputs ->
    identical replicates (the seeds are fixed), so reuse changes no number."""
    import hashlib
    trows = [r for r in rows if r["task"] == task]
    key = hashlib.sha256(json.dumps([sorted((r["arm"], r["w"], r["k"], r["label"], r["passed"]) for r in trows),
                                     sorted(classes), n_boot]).encode()).hexdigest()
    cp = RESULTS_DIR / "boot_cache" / f"{task}.json"
    if cp.exists():
        c = read_json(cp)
        if c.get("key") == key:
            return c["replicates"]
    out = _bootstrap(trows, task, classes, n_boot, n_workers)
    write_json(cp, {"key": key, "replicates": out})
    return out


def _bootstrap(trows, task: str, classes: list[str], n_boot: int, n_workers: int) -> list[dict[str, float]]:
    seeds = [BOOT_SEED5 * 1000 + i + 7919 * TASKS15.index(task) for i in range(n_boot)]
    if n_workers <= 1:  # notebook: a function defined in a notebook cannot be sent to "spawn" worker processes
        return _boot_worker((trows, task, classes, seeds))
    chunks = [seeds[i::n_workers] for i in range(n_workers)]
    with ProcessPoolExecutor(max_workers=n_workers, mp_context=mp.get_context("spawn")) as ex:
        parts = list(ex.map(_boot_worker, [(trows, task, classes, c) for c in chunks if c]))
    out = [x for p in parts for x in p]
    return out


def ci(vals: list[float]) -> list[float | None]:
    v = [x for x in vals if x is not None and np.isfinite(x)]
    if not v:
        return [None, None]
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]

## 8. The committed Claim-C rule (`claim_c_point`)

For each decisive task t:

- best_text(t) = the maximum H5 over the four text-level samplers {IFG, PSL, EAP, BINT}.
- PN **wins** t if round(H5_PN, 2) ≥ round(best_text, 2).

Claim C is SUPPORTED if PN wins ≥ 3 of the 5 decisive tasks. If fewer tasks are evaluable, as in this demo with
2 tasks, the threshold becomes ⌈0.6 · n⌉ = 2.

In [ ]:
def claim_c_point(h5: dict[str, dict[str, float]], text_classes: list[str], pn_class: str = "PN",
                  tasks: tuple[str, ...] = D5, min_wins: int | None = None) -> dict[str, Any]:
    per = {}
    wins = 0
    n_eval = 0
    for t in tasks:
        hp = h5.get(t, {}).get(pn_class)
        tx = {c: h5.get(t, {}).get(c) for c in text_classes if h5.get(t, {}).get(c) is not None}
        if hp is None or not tx:
            per[t] = {"evaluable": False}
            continue
        n_eval += 1
        best_c = max(tx, key=lambda c: tx[c])
        best = tx[best_c]
        win = round(hp, 2) >= round(best, 2)
        wins += win
        per[t] = {"H5_PN": hp, "best_text": best, "best_text_class": best_c, "text_H5": tx, "PN_wins": bool(win),
                  "tie": round(hp, 2) == round(best, 2), "D_t": hp - best}
    thr = min_wins if min_wins is not None else (CLAIM_C_MIN_WINS if n_eval == len(D5) else
                                                 math.ceil(0.6 * n_eval))
    verdict = "SUPPORTED" if wins >= thr else "DISCONFIRMED"
    dts = [v["D_t"] for v in per.values() if v.get("evaluable", True) and "D_t" in v]
    return {"per_task": per, "wins": wins, "n_tasks_evaluable": n_eval, "threshold": thr, "verdict": verdict,
            "D_bar": float(np.mean(dts)) if dts else None}

## 9. Run the analysis — 8.2: per-(task, arm) cells

From here on, the cells are the body of `analysis_c.run()`, split at its section markers. The first section loads the
rows, fixes the defaults and describes every (task, arm) cell.

In [ ]:
n_workers, n_boot_d5, n_boot_nd = N_WORKERS, N_BOOT_D5, N_BOOT_ND  # run()'s arguments
rows = load_rows()
idx = index(rows)
dfl = defaults(idx)
write_json(RESULTS_DIR / "defaults.json", dfl)
# ---------------------------------------------------------------- 8.2 cells
cells = {}
for (t, a), cell in sorted(idx.items()):
    cells[f"{t}|{a}"] = cell_stats(list(cell.values()), dfl.get(t, {}).get("default"))
write_json(RESULTS_DIR / "cells.json", cells)
write_json(RESULTS_DIR / "incomplete_cells.json",
           {k: v["N"] for k, v in cells.items() if v["N"] < min_n(k.split("|")[0])})

print(len(rows), "labelled rows; defaults:", {t: dfl[t]["default"] for t in data["metadata"]["tasks"]})
display(pd.DataFrame([{"cell": k, "N": v["N"], "pass": v["pass"], "Hill (depth 25)": v["hill_rarefied"],
                       "off-default share": v["off_default_share"], "correct composition": v["correct_composition"]}
                      for k, v in cells.items()]).round(3))

## 10. Section 8.3: point prices on the hull

Every class of every task is priced on the full simplex (`keep_frontier` for figure F2). The edge-restricted hull is
priced too, as a sensitivity. With `N_DRAWS_PRICE = 1000` the H5 / P15 / P20 values equal the committed
`results/prices.json`.

In [ ]:
t0 = time.time()
# ---------------------------------------------------------------- 8.3 point prices
prices: dict[str, dict[str, Any]] = defaultdict(dict)
for t in TASKS15:
    for cls in CLASS_SETTINGS:
        p = price(idx, t, cls, n_draws=N_DRAWS_PRICE, seed=0, keep_frontier=True)
        if p is not None:
            pe = price(idx, t, cls, n_draws=N_DRAWS_PRICE, seed=0, max_nonzero=2)
            p["H5_edge"] = pe["H5"] if pe else None
            p["full_ge_edge"] = (p["H5"] >= pe["H5"] - 1e-9) if pe else None
            prices[t][cls] = p
write_json(RESULTS_DIR / "prices.json", prices)
h5 = {t: {c: v["H5"] for c, v in d.items()} for t, d in prices.items()}
h5_edge = {t: {c: v.get("H5_edge") for c, v in d.items()} for t, d in prices.items()}
print(f"point prices for {len(prices)} tasks in {time.time() - t0:.0f}s")

ref = data["metadata"]["reference"]
display(pd.DataFrame([{"task": t, "class": c, "arms": "+".join(v["arms"]), "H5": round(v["H5"], 4),
                       "H5 committed": round(ref["knob_table"][t][c]["H5"], 4), "P15": round(v["P15"], 4) + 0.0,
                       "P20": round(v["P20"], 4) + 0.0,
                       "H5 edge-only": round(v["H5_edge"], 4), "best mixture w": [round(x, 3) for x in v["H5_mixture"]]}
                      for t, d in prices.items() for c, v in d.items()]))

## 11. Section 8.3: slot bootstrap

This section runs `N_BOOT_D5` replicates per task. Each replicate re-prices every class, and the replicates feed the
confidence intervals and P(PN ≥ best text sampler).

In [ ]:
t0 = time.time()
# ---------------------------------------------------------------- 8.3 bootstrap
boots: dict[str, list[dict[str, float]]] = {}
for t in TASKS15:
    classes = [c for c in CLAIM_CLASSES + OTHER_CLASSES if c in prices.get(t, {})]
    nb = n_boot_d5 if t in D5 else n_boot_nd
    if classes:
        boots[t] = bootstrap(rows, t, classes, nb, n_workers)
write_json(RESULTS_DIR / "bootstrap.json", boots)
print(f"{ {t: len(b) for t, b in boots.items()} } replicates in {time.time() - t0:.0f}s")

## 12. Section 8.4: the Claim-C verdict

This section applies the committed rule to the point H5 values. Beside the verdict it reports:

- the bootstrap probability P(PN ≥ best text sampler) per task, flagged UNRESOLVED if it lies in [0.2, 0.8];
- the replicate distribution of the win count;
- D̄ = mean_t (H5_PN − best_text), with its percentile CI;
- the sensitivities.

The two audit-based sensitivities need `audit/style_regate.json` and `audit/novelty_relabel.json`. These files are
not in the demo data, so their `exists()` checks skip them. In the full run both gave DISCONFIRMED, 1/5.

In [ ]:
# ---------------------------------------------------------------- 8.4 Claim C
primary = claim_c_point(h5, list(TEXT_SAMPLERS))
rep_wins, rep_dbar = [], []
p_win = {}
for t in D5:
    bt = boots.get(t, [])
    ok = []
    for b in bt:
        tx = [b[c] for c in TEXT_SAMPLERS if c in b and b[c] is not None and np.isfinite(b[c])]
        if "PN" in b and tx:
            ok.append(round(b["PN"], 2) >= round(max(tx), 2))
    p_win[t] = float(np.mean(ok)) if ok else None
nrep = min((len(boots.get(t, [])) for t in D5 if t in boots), default=0)  # notebook: only the demo's D5 tasks
for i in range(nrep):
    w_, d_ = 0, []
    for t in D5:
        if t not in boots:  # notebook: skip the decisive tasks that are not in the demo data
            continue
        b = boots[t][i]
        tx = [b[c] for c in TEXT_SAMPLERS if c in b and b[c] is not None and np.isfinite(b[c])]
        if "PN" in b and tx:
            w_ += round(b["PN"], 2) >= round(max(tx), 2)
            d_.append(b["PN"] - max(tx))
    rep_wins.append(w_)
    if d_:
        rep_dbar.append(float(np.mean(d_)))
sens = {
    "IFG_hot_only": claim_c_point(h5, ["IFG_hot_only", "PSL", "EAP", "BINT"]),
    "PN_plus": claim_c_point(h5, list(TEXT_SAMPLERS), pn_class="PN+"),
    "edge_restricted_hull": claim_c_point(h5_edge, list(TEXT_SAMPLERS)),
    "leftpad_arms_dropped": claim_c_point(h5, ["IFG"]),
}
flags_p = AUDIT_DIR / "style_regate.json"
if flags_p.exists():
    fl = read_json(flags_p).get("flagged_tasks", [])
    keep = tuple(t for t in D5 if t not in fl)
    sens["style_regate_flagged_excluded"] = {**claim_c_point(h5, list(TEXT_SAMPLERS), tasks=keep),
                                             "excluded": fl}
nov_p = AUDIT_DIR / "novelty_relabel.json"
if nov_p.exists():
    relabel = read_json(nov_p).get("relabel", {})
    idx2 = index(load_rows(relabel))
    h5r = {t: {c: (lambda p: p["H5"] if p else None)(price(idx2, t, c, n_draws=N_DRAWS_PRICE, seed=0))
               for c in ["PN"] + list(TEXT_SAMPLERS)} for t in D5}
    h5r = {t: {c: v for c, v in d.items() if v is not None} for t, d in h5r.items()}
    sens["novelty_audit_relabelled"] = claim_c_point(h5r, list(TEXT_SAMPLERS))
claim = {"primary": primary, "verdict": primary["verdict"],
         "p_PN_ge_best_text": p_win,
         "unresolved": {t: (p is not None and UNRESOLVED_BAND[0] <= p <= UNRESOLVED_BAND[1]) for t, p in p_win.items()},
         "replicate_win_count_distribution": dict(Counter(rep_wins)), "n_replicates": nrep,
         "D_bar": primary["D_bar"], "D_bar_ci95": ci(rep_dbar),
         "class_ci95": {t: {c: ci([b.get(c) for b in boots.get(t, [])]) for c in CLAIM_CLASSES + OTHER_CLASSES}
                        for t in TASKS15},
         "sensitivities": sens,
         "rule": ("PN wins t if round(H5_PN,2) >= round(max_{IFG,PSL,EAP,BINT} H5,2); SUPPORTED if wins >= 3 of 5 "
                  "(D5 = T03, T10, T2, T05, T1), else DISCONFIRMED")}
nd = {t: claim_c_point(h5, list(TEXT_SAMPLERS), tasks=(t,), min_wins=1)["per_task"].get(t) for t in ND10}
claim["nd10_descriptive"] = nd
write_json(RESULTS_DIR / "claim_c.json", claim)

for t, v in primary["per_task"].items():
    if v.get("evaluable", True):
        print(f"{t}: H5_PN {v['H5_PN']:.2f} vs best text {v['best_text_class']} {v['best_text']:.2f} -> "
              f"{'PN wins' if v['PN_wins'] else 'PN loses'};  P(PN >= best) = {p_win[t]}")
print(f"demo verdict: PN wins {primary['wins']} of {primary['n_tasks_evaluable']} evaluable decisive tasks "
      f"(threshold {primary['threshold']}) -> {claim['verdict']};  D_bar = {primary['D_bar']:.3f}, "
      f"95% CI {[round(x, 2) for x in claim['D_bar_ci95'] if x is not None]}")
print(f"committed (all 5 decisive tasks, 2,000 replicates): PN wins {ref['wins_of_5']} of 5 -> {ref['verdict_C']}; "
      f"D_bar = {ref['D_bar']:.2f}, 95% CI {[round(x, 2) for x in ref['D_bar_ci95']]}")
print("sensitivities:", {k: f"{v['verdict']} ({v['wins']}/{v['n_tasks_evaluable']})" for k, v in sens.items()})

## 13. Section 8.5: plan noise vs departure-matched decoding

Does PN move the algorithm only because it perturbs the first tokens? DMD departs from TWIN by the same per-token KL,
by temperature alone. For each slot the indicator is "correct and off-default". The exact McNemar test compares PN_a
with DMD_a on the discordant slots, and a pooled exact test sums the discordant slots over the decisive tasks.

In [ ]:
# ---------------------------------------------------------------- 8.5 PN vs DMD
dmd = data["metadata"]["dmd_temperatures"]  # notebook: was predictions/dmd_temperatures.json
pvd: dict[str, Any] = {"per_task": {}}
pooled = defaultdict(lambda: [0, 0])
for t in TASKS15:
    d = dfl.get(t, {}).get("default")
    tw = idx.get((t, "TWIN"), {})
    rec = {}
    for alpha in ("1", "2"):
        pa, da = idx.get((t, f"PN_a{alpha}"), {}), idx.get((t, f"DMD_a{alpha}"), {})
        common = sorted(set(pa) & set(da) & set(tw))
        if not common or d is None:
            continue
        ind = lambda cell: [int(cell[s]["passed"] and cell[s]["label"] != d) for s in common]
        x, y, z = ind(pa), ind(da), ind(tw)
        m = mcnemar_exact(x, y)
        if t in D5:
            pooled[alpha][0] += m["x_only"]
            pooled[alpha][1] += m["y_only"]
        cal = dmd.get(t, {}).get(f"{float(alpha)}", {})
        rec[f"alpha{alpha}"] = {
            "n_slots": len(common), "PN_off_default_share": float(np.mean(x)), "DMD_off_default_share": float(np.mean(y)),
            "TWIN_off_default_share": float(np.mean(z)),
            "PN_hill": cells.get(f"{t}|PN_a{alpha}", {}).get("hill_rarefied"),
            "DMD_hill": cells.get(f"{t}|DMD_a{alpha}", {}).get("hill_rarefied"),
            "PN_pass": cells.get(f"{t}|PN_a{alpha}", {}).get("pass"),
            "DMD_pass": cells.get(f"{t}|DMD_a{alpha}", {}).get("pass"),
            "mcnemar_PN_vs_DMD": m, "mcnemar_PN_vs_TWIN": mcnemar_exact(x, z), "mcnemar_DMD_vs_TWIN": mcnemar_exact(y, z),
            "K_PN": cal.get("K_PN"), "f_on_DMD": cal.get("f_on"), "T_dmd": cal.get("T_dmd"),
            "token_departure_PN": cal.get("token_departure_PN"), "token_departure_DMD": cal.get("token_departure_DMD")}
    rec["H5_PN"] = h5.get(t, {}).get("PN")
    rec["H5_DMD"] = h5.get(t, {}).get("DMD")
    pvd["per_task"][t] = rec
from scipy.stats import binomtest
pvd["pooled_D5_exact"] = {a: {"PN_only": b, "DMD_only": c, "p_exact": float(binomtest(b, b + c, 0.5).pvalue)
                              if b + c else 1.0} for a, (b, c) in pooled.items()}
write_json(RESULTS_DIR / "pn_vs_dmd.json", pvd)

for t, rec in pvd["per_task"].items():
    for a in ("alpha1", "alpha2"):
        if a in rec:
            m, r_ = rec[a]["mcnemar_PN_vs_DMD"], ref["pn_vs_dmd"][t][a]["mcnemar_PN_vs_DMD"]
            print(f"{t} {a}: T_dmd {rec[a]['T_dmd']:.2f} | off-default share PN {rec[a]['PN_off_default_share']:.2f} "
                  f"DMD {rec[a]['DMD_off_default_share']:.2f} TWIN {rec[a]['TWIN_off_default_share']:.2f} | "
                  f"PN-only/DMD-only {m['x_only']}/{m['y_only']} p={m['p_exact']:.4g} "
                  f"(committed {r_['x_only']}/{r_['y_only']})")
print("pooled over the demo tasks:", pvd["pooled_D5_exact"])

## 14. Section 8.6: PN on the plain prompt vs PN on the twin

PNPLAIN_a1 and PN_a1 share the plan-noise direction and the token stream at every slot, so they are paired by slot.
PLAIN vs TWIN is paired the same way.

In [ ]:
# ---------------------------------------------------------------- 8.6 PN plain vs PN twin
pvt = {}
for t in TASKS15:
    d = dfl.get(t, {}).get("default")
    if d is None:
        continue
    rec = {}
    for a_x, a_y in (("PNPLAIN_a1", "PN_a1"), ("PLAIN", "TWIN")):
        cx, cy = idx.get((t, a_x), {}), idx.get((t, a_y), {})
        common = sorted(set(cx) & set(cy))
        if not common:
            continue
        ix = [int(cx[s]["passed"] and cx[s]["label"] != d) for s in common]
        iy = [int(cy[s]["passed"] and cy[s]["label"] != d) for s in common]
        rec[f"{a_x}_vs_{a_y}"] = {"n_slots": len(common), "mcnemar": mcnemar_exact(ix, iy),
                                  "pass_x": float(np.mean([cx[s]["passed"] for s in common])),
                                  "pass_y": float(np.mean([cy[s]["passed"] for s in common])),
                                  "hill_x": cells.get(f"{t}|{a_x}", {}).get("hill_rarefied"),
                                  "hill_y": cells.get(f"{t}|{a_y}", {}).get("hill_rarefied")}
    pvt[t] = rec
write_json(RESULTS_DIR / "plain_vs_twin.json", pvt)

for t, rec in pvt.items():
    print(t, {k: f"{v['mcnemar']['x_only']}/{v['mcnemar']['y_only']} (p={v['mcnemar']['p_exact']:.2g}), "
                 f"pass {v['pass_x']:.2f} vs {v['pass_y']:.2f}" for k, v in rec.items()})

## 15. Sections 8.7–8.9: unit ratio, landings and the knob table

- **8.7.** The ban-free / ban unit ratio needs the per-row activation-norm ratios and exp19's FP8 H5 values. Neither
  is in the demo data, so only H5_PN is filled.
- **8.8, landings.** A knob *moves* a task if its correct-only off-default share is ≥ TWIN's + 0.10, with a one-sided
  Fisher p < 0.05. Its *landing set* is where the moved programs go, compared with the iteration-1 Qwen3-8B-Base
  compositions of T1/T2, which are in the demo data.
  - Does a knob only reach algorithms the base model already writes, or something outside that support?
  - The EAP list tally needs the artifact's `parsers` module and the raw enumeration lists, so it records an
    ImportError here.
  - In the demo, T2's landing sets (`bf_spfa`, `floyd`) count as 100% outside the base support. That is only because
    iteration 1 spells the same algorithms differently (`bellman_ford_spfa`, `floyd_warshall`). The artifact's
    post-hoc label harmonisation (amendment A4) gives 0 for every T2 landing.
- **8.9, knob table.** The table collects H5 / P15 / P20 with their bootstrap CIs.

In [ ]:
# ---------------------------------------------------------------- 8.7 unit ratio
ur = {}
e19c1 = {}
try:
    c1 = read_json(INPUTS / "exp19" / "results" / "c1.json")
    for t, v in (c1.get("primary", {}).get("per_task", {}) or {}).items():
        e19c1[t] = v.get("H5_PN")
except (FileNotFoundError, KeyError, AttributeError):
    pass
for t in TASKS15:
    rs = [r for r in rows if r["task"] == t and r["arm"] in ("PN_a1", "PN_a2") and r.get("unit_ratio")]
    ratios = [float(np.mean(list(r["unit_ratio"].values()))) for r in rs]
    ur[t] = {"banfree_over_ban_mean": float(np.mean(ratios)) if ratios else None, "n": len(ratios),
             "H5_PN_banfree_bf16": h5.get(t, {}).get("PN"), "H5_PN_ban_unit_exp19_fp8": e19c1.get(t),
             "note": "exp19 rows are FP8 W8A8 (other engine); descriptive only"}
write_json(RESULTS_DIR / "unit_ratio.json", ur)
# ---------------------------------------------------------------- 8.8 landings
from scipy.stats import fisher_exact
base = defaultdict(Counter)
for t_, comp_ in data["metadata"]["base_compositions_iter1"].items():  # notebook: was the exp5 rows, tallied
    base[t_].update(comp_)
land: dict[str, Any] = {"moves": {}, "base_compositions_iter1": {t: dict(base[t]) for t in ("T1", "T2")}}
for (t, a), cell in sorted(idx.items()):
    d = dfl.get(t, {}).get("default")
    if a == "TWIN" or d is None or (t, "TWIN") not in idx:
        continue
    tw = list(idx[(t, "TWIN")].values())
    rs = list(cell.values())
    cx = [r for r in rs if r["passed"]]
    ct = [r for r in tw if r["passed"]]
    if not cx or not ct:
        continue
    ox = sum(r["label"] != d for r in cx)
    ot = sum(r["label"] != d for r in ct)
    sx, st_ = ox / len(cx), ot / len(ct)
    p = float(fisher_exact([[ox, len(cx) - ox], [ot, len(ct) - ot]], alternative="greater")[1])
    moves = (sx >= st_ + MOVE_EXCESS) and p < MOVE_FISHER_P
    rec = {"correct_only_off_default": sx, "twin_correct_only_off_default": st_, "fisher_p_one_sided": p,
           "moves": bool(moves)}
    if moves:
        ls = Counter(r["label"] for r in cx if r["label"] != d)
        rec["landing_set"] = dict(ls)
        if t in ("T1", "T2"):
            sup = set(base[t])
            outside = sum(v for k, v in ls.items() if k not in sup)
            rec["base_support"] = sorted(sup)
            rec["landing_mass_outside_base_support"] = outside / max(sum(ls.values()), 1)
        rec["base_repertoire_step1"] = None
    land["moves"][f"{t}|{a}"] = rec
eap = {}
try:
    from parsers import map_modes
    import prompts_v4 as P4
    from cfg import BENCH
    for t in TASKS15:
        rs = [r for r in rows if r["task"] == t and r["arm"] == "EAP" and r.get("enum_list")]
        if not rs:
            continue
        al = {}
        if t not in ("T1", "T2"):
            sp = read_json(BENCH / P4.bench_dir(t) / "spec.json")
            al = {m: list(v) + [c["display_name"] for c in sp["catalogue"] if c["mode"] == m]
                  for m, v in sp.get("aliases", {}).items()}
        else:
            al = {"dijkstra": ["dijkstra"], "bellman_ford": ["bellman-ford", "bellman ford"], "spfa": ["spfa"],
                  "bfs": ["bfs", "breadth-first", "breadth first"], "dfs": ["dfs", "depth-first", "depth first"],
                  "a_star": ["a*", "a-star", "a star"], "bidirectional": ["bidirectional"],
                  "floyd_warshall": ["floyd", "warshall"]}
        comp = Counter()
        match = []
        for r in rs:
            for it in r["enum_list"]:
                ms = map_modes(it["name"], it.get("idea", ""), al)
                comp.update(ms or ["unmapped"])
            pk = next((it for it in r["enum_list"] if it["name"] == r.get("picked_name")), None)
            if pk is not None and r["passed"]:
                ms = map_modes(pk["name"], pk.get("idea", ""), al)
                match.append(r["label"] in ms if ms else None)
        mm = [m for m in match if m is not None]
        eap[t] = {"list_mode_composition": dict(comp), "n_lists": len(rs),
                  "mean_list_len": float(np.mean([len(r["enum_list"]) for r in rs])),
                  "picked_matches_label_rate": float(np.mean(mm)) if mm else None, "n_match_eval": len(mm)}
except (ImportError, FileNotFoundError, KeyError) as e:
    eap["error"] = str(e)
land["eap"] = eap
write_json(RESULTS_DIR / "landings.json", land)
# ---------------------------------------------------------------- 8.9 knob table
kt = {}
for t in TASKS15:
    kt[t] = {}
    for cls, p in prices.get(t, {}).items():
        bt = boots.get(t, [])
        kt[t][cls] = {"H5": p["H5"], "H5_ci95": ci([b.get(cls) for b in bt]), "P15": p["P15"],
                      "P15_ci95": ci([b.get(cls + "_P15") for b in bt]), "P20": p["P20"],
                      "P20_ci95": ci([b.get(cls + "_P20") for b in bt]), "arms": p["arms"],
                      "engine": "hf_bf16 (this artifact)", "n_boot": len(bt)}
write_json(RESULTS_DIR / "knob_table.json", kt)

print("unit ratio:", {t: v for t, v in ur.items() if v["H5_PN_banfree_bf16"] is not None})
print("moving knobs:", {k: v.get("landing_set") for k, v in land["moves"].items() if v["moves"]})
print("T2 landing mass outside the iteration-1 base support:",
      {k: v["landing_mass_outside_base_support"] for k, v in land["moves"].items()
       if v["moves"] and "landing_mass_outside_base_support" in v})
print("EAP list tally:", land["eap"])

## 16. Results

### 16.1 Knob table: this notebook vs the committed full run

H5 is the largest correct-only diversity reachable within a 5-point pass budget. 1.0 means every correct program uses
the default algorithm.

The demo's bootstrap CIs use `N_BOOT_D5` replicates, against 2,000 in the committed run.

In [ ]:
kt_rows = []
for t, d in RESULTS["knob_table.json"].items():
    if not d:
        continue
    for c, v in d.items():
        r = ref["knob_table"][t][c]
        kt_rows.append({"task": t, "class": c, "H5": round(v["H5"], 2), "H5 committed": round(r["H5"], 2),
                        "H5 95% CI": [round(x, 2) for x in v["H5_ci95"]] if v["H5_ci95"][0] is not None else None,
                        "H5 95% CI committed": [round(x, 2) for x in r["H5_ci95"]] if r["H5_ci95"][0] is not None else None,
                        "P15": round(v["P15"], 3) + 0.0, "P20": round(v["P20"], 3) + 0.0,  # + 0.0 turns -0.0 into 0.0
                        "n_boot": v["n_boot"]})
kt_df = pd.DataFrame(kt_rows)
display(kt_df)
print("max |H5 - committed H5| =", float((kt_df["H5"] - kt_df["H5 committed"]).abs().max()))

### 16.2 Figures (`src/figures5.py`)

These are the original figure functions. They read the in-memory results instead of `results/*.json` and show the
figures inline instead of saving them. The panels cover the demo tasks.

- **F1 — the literal cell.** Drawn from the committed Claim-F result in the demo data's metadata.
- **F2 — hull frontiers.** The 5-point budget is the dotted line.
- **F3 — compositions.** Each arm's correct-label composition.
- **F4 — PN vs DMD.** PN's per-token KL curve and the off-default shares.
- **F5 — PN plain vs PN twin.**
- **F6 — landings.** The landing sets against the base model's composition.

In [ ]:
# src/figures5.py
plt.rcParams.update({"font.size": 9, "pdf.fonttype": 42, "ps.fonttype": 42, "axes.spines.top": False,
                     "axes.spines.right": False})
PAL = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
CLS = ["PN", "IFG", "PSL", "EAP", "BINT", "DMD"]


def save(fig, name: str) -> None:
    plt.show()  # notebook: inline instead of figures/<name>.pdf/.png


def f1() -> None:
    lc = data["metadata"]["literal_cell"]  # notebook: the committed results/literal_cell.json
    grid = lc["seed_grid"]
    Ts = ["0.3", "0.7", "1.0", "1.3"]
    labs = sorted({c["labels"][t] for c in grid for t in Ts if c["labels"][t]})
    col = {l: PAL[i % len(PAL)] for i, l in enumerate(labs)}
    fig, ax = plt.subplots(figsize=(4.2, 6.0))
    for i, c in enumerate(grid):
        for j, t in enumerate(Ts):
            lab = c["labels"][t]
            ax.add_patch(plt.Rectangle((j, i), 0.95, 0.9, color=col.get(lab, "#eeeeee")))
            fa = (c["first_approach"][t] or "-")[:4]
            ax.text(j + 0.47, i + 0.45, fa + ("" if c["completed"][t] else "*"), ha="center", va="center", fontsize=6,
                    color="white")
    ax.set_xlim(0, 4)
    ax.set_ylim(0, len(grid))
    ax.set_xticks(np.arange(4) + 0.47, [f"T={t}" for t in Ts])
    ax.set_yticks(np.arange(len(grid)) + 0.45, [c["seed"] for c in grid], fontsize=6)
    ax.invert_yaxis()
    ct = lc["C_T"]
    ax.set_title(f"Literal cell (T2, thinking on, top-k 10): final label per seed x T\n"
                 f"C_T = {ct['n_change_final_label']}/{ct['n_seeds_all4_completed']} "
                 f"(CP one-sided 95% UB {ct['clopper_pearson']['upper95_one_sided']:.3f}); text = first approach",
                 fontsize=8)
    handles = [plt.Rectangle((0, 0), 1, 1, color=col[l]) for l in labs]
    ax.legend(handles, labs, loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=3, fontsize=7, frameon=False)
    save(fig, "F1_literal_cell")


def f2() -> None:
    pr = RESULTS["prices.json"]
    fig, axs = plt.subplots(1, len(FIG_TASKS), figsize=(3.0 * len(FIG_TASKS), 2.8), sharey=True)
    for ax, t in zip(axs, FIG_TASKS):
        for i, c in enumerate(CLS):
            p = pr.get(t, {}).get(c)
            if not p or not p.get("frontier"):
                continue
            fr = np.array(p["frontier"])
            ax.step(fr[:, 0] * 100, fr[:, 1], where="post", color=PAL[i], label=c, lw=1.4)
        ax.axvline(5, color="#999", ls=":", lw=0.8)
        ax.set_title(t)
        ax.set_xlabel("pass cost (points)")
        ax.set_xlim(-30, 40)
    axs[0].set_ylabel("correct-only Hill-1 (depth 25)")
    axs[-1].legend(fontsize=7, frameon=False)
    fig.suptitle("Randomisation-hull frontiers per decisive task (dotted line: the 5-point budget of H5)", fontsize=9,
                 y=1.04)
    save(fig, "F2_hull_frontiers")


def f3() -> None:
    cells = RESULTS["cells.json"]
    arms = ["TWIN", "PN_a1", "PN_a2", "DMD_a1", "DMD_a2", "IFG_hot", "IFG_paper", "PSL", "EAP", "BINT", "T1.3", "NOVELTY"]
    fig, axs = plt.subplots(len(FIG_TASKS), 1, figsize=(8.5, 2.0 * len(FIG_TASKS)))
    for ax, t in zip(axs, FIG_TASKS):
        present = [a for a in arms if f"{t}|{a}" in cells]
        labs = sorted({l for a in present for l in cells[f"{t}|{a}"]["correct_composition"]})
        col = {l: PAL[i % len(PAL)] for i, l in enumerate(labs)}
        for j, a in enumerate(present):
            c = cells[f"{t}|{a}"]
            bottom = 0.0
            for l in labs:
                v = c["correct_composition"].get(l, 0) / max(c["N"], 1)
                if v:
                    ax.bar(j, v, bottom=bottom, color=col[l], width=0.8)
                    bottom += v
        ax.set_xticks(range(len(present)), present, fontsize=7)
        ax.set_ylabel(f"{t}\nshare of N", fontsize=8)
        ax.legend([plt.Rectangle((0, 0), 1, 1, color=col[l]) for l in labs], labs, fontsize=6, ncol=4,
                  loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
    fig.suptitle("Correct-only compositions by arm (bar height = pass rate)", fontsize=9)
    save(fig, "F3_compositions")


def f4() -> None:
    dmd = data["metadata"]["dmd_temperatures"]
    pvd = RESULTS["pn_vs_dmd.json"]["per_task"]
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.0))
    for i, t in enumerate(FIG_TASKS):
        for a, ls in (("1.0", "-"), ("2.0", "--")):
            v = dmd.get(t, {}).get(a)
            if v:
                axs[0].plot(range(1, len(v["kl_curve_PN"]) + 1), v["kl_curve_PN"], ls=ls, color=PAL[i],
                            label=f"{t} a{a[0]}" if a == "1.0" else None)
    axs[0].set_xlabel("generated token")
    axs[0].set_ylabel("KL(p_push || p_clean)")
    axs[0].set_title("PN per-token departure (calibration rows; dashed = alpha 2)", fontsize=8)
    axs[0].legend(fontsize=6, frameon=False)
    x = np.arange(len(FIG_TASKS))
    wd = 0.2
    for k, (key, lab, c) in enumerate((("PN_off_default_share", "PN", PAL[0]), ("DMD_off_default_share", "DMD", PAL[5]))):
        for a, off in (("alpha1", -1.5), ("alpha2", 0.5)):
            vals = [pvd.get(t, {}).get(a, {}).get(key, np.nan) for t in FIG_TASKS]
            axs[1].bar(x + (off + k) * wd, vals, width=wd, color=c, alpha=1.0 if a == "alpha1" else 0.55,
                       label=f"{lab} {a}")
    axs[1].set_xticks(x, FIG_TASKS)
    axs[1].set_ylabel("correct off-default share of N")
    axs[1].set_title("PN vs departure-matched decoding", fontsize=8)
    axs[1].legend(fontsize=6, frameon=False)
    save(fig, "F4_pn_vs_dmd")


def f5() -> None:
    pvt = RESULTS["plain_vs_twin.json"]
    ts = [t for t in pvt if "PNPLAIN_a1_vs_PN_a1" in pvt[t]]
    fig, ax = plt.subplots(figsize=(6, 2.6))
    x = np.arange(len(ts))
    a = [pvt[t]["PNPLAIN_a1_vs_PN_a1"]["mcnemar"]["x_only"] for t in ts]
    b = [pvt[t]["PNPLAIN_a1_vs_PN_a1"]["mcnemar"]["y_only"] for t in ts]
    ax.bar(x - 0.2, a, 0.4, color=PAL[2], label="off-default only under PN plain")
    ax.bar(x + 0.2, b, 0.4, color=PAL[0], label="off-default only under PN twin")
    for i, t in enumerate(ts):
        p = pvt[t]["PNPLAIN_a1_vs_PN_a1"]["mcnemar"]["p_exact"]
        ax.text(i, max(a[i], b[i]) + 0.3, f"p={p:.2f}", ha="center", fontsize=6)
    ax.set_xticks(x, ts, fontsize=7)
    ax.set_ylabel("discordant slots")
    ax.set_title("PN plain vs PN twin, paired by direction and slot (exact McNemar)", fontsize=8)
    ax.legend(fontsize=6, frameon=False)
    save(fig, "F5_plain_vs_twin")


def f6() -> None:
    land = RESULTS["landings.json"]
    base = land["base_compositions_iter1"]
    fig, axs = plt.subplots(1, 2, figsize=(9, 2.8))
    for ax, t in zip(axs, ("T1", "T2")):
        sets = {"base (iter 1)": Counter(base.get(t, {}))}
        for k, v in land["moves"].items():
            if k.startswith(t + "|") and v.get("moves"):
                sets[k.split("|")[1]] = Counter(v["landing_set"])
        labs = sorted({l for c in sets.values() for l in c})
        col = {l: PAL[i % len(PAL)] for i, l in enumerate(labs)}
        for j, (nm, c) in enumerate(sets.items()):
            tot = max(sum(c.values()), 1)
            bottom = 0.0
            for l in labs:
                v = c.get(l, 0) / tot
                if v:
                    ax.bar(j, v, bottom=bottom, color=col[l])
                    bottom += v
        ax.set_xticks(range(len(sets)), list(sets), rotation=30, fontsize=7)
        ax.set_title(f"{t}: landing sets of moving knobs vs base composition", fontsize=8)
        ax.legend([plt.Rectangle((0, 0), 1, 1, color=col[l]) for l in labs], labs, fontsize=6, frameon=False,
                  loc="upper left", bbox_to_anchor=(1.0, 1.0))
    save(fig, "F6_landings")


FIG_TASKS = tuple(t for t in D5 if RESULTS["prices.json"].get(t))  # notebook: the decisive tasks in the demo data
for f in (f1, f2, f3, f4, f5, f6):
    try:
        f()
        print(f"{f.__name__} ok")
    except (FileNotFoundError, KeyError, ValueError, TypeError) as e:
        print(f"{f.__name__} skipped: {e}")

### 16.3 Claim F, re-checked from the committed seed grid (`src/analysis_f.py`)

C_T is the number of seeds whose final-code label is not constant across T ∈ {0.3, 0.7, 1.0, 1.3}, among the seeds
with all four temperatures completed. It comes with the exact Clopper-Pearson bound. This uses the original
`clopper_pearson` and the C_T count of `analysis_f.run()`, applied to the committed per-seed label grid.

In [ ]:
# src/analysis_f.py
F_TEMPS = (0.3, 0.7, 1.0, 1.3)


def clopper_pearson(x: int, n: int) -> dict[str, float | None]:
    from scipy.stats import beta

    if n == 0:
        return {"upper95_one_sided": None, "ci95_two_sided": [None, None]}
    up1 = 1.0 if x == n else float(beta.ppf(0.95, x + 1, n - x))
    lo2 = 0.0 if x == 0 else float(beta.ppf(0.025, x, n - x + 1))
    up2 = 1.0 if x == n else float(beta.ppf(0.975, x + 1, n - x))
    return {"upper95_one_sided": up1, "ci95_two_sided": [lo2, up2]}


lc = data["metadata"]["literal_cell"]
full = [c for c in lc["seed_grid"] if all(c["completed"][f"{t:.1f}"] for t in F_TEMPS)]  # all 4 T completed


def count_change(cells: list[dict[str, Any]], field: str) -> int:
    return sum(1 for c in cells if len(set(c[field].values())) > 1)

ct = count_change(full, "labels")
cfa = count_change(full, "first_approach")
print(f"C_T = {ct}/{len(full)} seeds change their final-code algorithm across T; first approach changes on {cfa}")
print("Clopper-Pearson:", clopper_pearson(ct, len(full)), "| prediction C_T <= 1 met:", ct <= lc["prediction"]["C_T_max"])
print("final labels over all seeds x T:", dict(Counter(v for c in lc["seed_grid"] for v in c["labels"].values())))
print("noise floor (T 0.7 vs its repeat):", lc["noise_floor"])
print("paired with exp21 (top-p 1, same seed):", {k: lc["paired_exp21"][k] for k in ("n_pairs", "label_agreement")},
      "median common token prefix:", lc["paired_exp21"]["common_prefix_tokens_quantiles"]["0.5"])

## Takeaways

- **Temperature is not the creativity knob, at least for algorithm choice.**
  - Across T ∈ {0.3 … 1.3}, 20/20 seeds keep Dijkstra on the shortest-path task.
  - The traces diverge after a median of only 34 tokens from the top-p-1 run, yet every label agrees. Temperature
    changes the *phrasing*, not the *idea*.
  - In the knob table, T1.3 and top-k 10 have H5 ≈ 1.0–1.09.
- **Activation-level plan noise does move the algorithm.**
  - On T03, PN's H5 is 1.98: about two algorithms at no pass cost, mostly quickselect next to full sort.
  - PN beats departure-matched decoding at α 2. On T03 the discordant slots split 13/0, so the effect is not just
    early-token noise.
- **As a practical knob, though, PN loses to text-level planning.**
  - On T2, a **base-model-written intent** (BINT) reaches H5 1.45 against PN's 1.18.
  - Over all five decisive tasks, PN wins only 1 → DISCONFIRMED.
  - Context that names a plan pushes the model into a different part of its repertoire more cheaply than noise in
    the residual stream.

**Scaling up.** Set `N_BOOT_D5 = 2000`, the original value: the bootstrap CIs and P(PN ≥ best) then equal the
committed ones, at ≈ 7–8 minutes on one CPU. The original used 12 worker processes. From a notebook, `N_WORKERS > 1`
needs the worker functions in an importable module. To
re-run generation itself, use the artifact's `method.py` on a ≥ 24 GB GPU (see its README).